# ZAIC Relearning v1 — synthetic CPU preflight

New experiment, baseline unchanged. Default: 99 baseline tests + new tests + synthetic correction/replay round. No contest data, no GPU. Warm-start GPU block stays gated by permission evidence and explicit smoke configuration. Stop session and set Accelerator=None immediately after GPU work. Full resume is separate and disabled.

Prepared revision adds CPU post-smoke validator tests, a 900-second child timeout, saved-adapter reload and native/masked loss parity. This revision has not run on Kaggle/GPU yet. No long training. Stop GPU session explicitly after work.

In [ ]:
CONFIG={'action': 'preflight', 'init_mode': 'warm_start', 'mode': 'learning', 'gpu_confirmed': False, 'competition_permission_evidence': {'status': 'unverified'}, 'max_steps': 1, 'approved_max_steps': 1, 'max_seconds': 180, 'approved_max_seconds': 180, 'lr': 5e-06, 'gradient_accumulation': 1, 'learning_businesses': ['synthetic'], 'revision': 'c202236235762e1c871ad0ccb60c8ee5ba337b9a', 'parent_sha256': 'e7dfa635beccce128c99f9d3fd993d33dbb4c88e73e3ddeccbc2bc915eec4a4d', 'user_authorized': True, 'base_hashes': {'README.md': 'c5f5a8c2dddab69cfbf05279235aa5fddb137939a06539c4c7637aa900fef6d0', 'LICENSE': 'bbedc3fda3305820b977265f01b8619d87570a6739de3a5582c3464840f1e57a', 'config.json': 'd0883072e01861ed0b2d47be3c16c36a8e81c224c7ffaa310c6558fb3f932b05', 'chat_template.jinja': 'a4aee8afcf2e0711942cf848899be66016f8d14a889ff9ede07bca099c28f715', 'merges.txt': 'a9d356d7bdf1ef4949e3e748e95b8e10ad9d4e2e838eddc38a0a7b6b94d1db8d', 'model.safetensors-00002-of-00004.safetensors': '31c7d7e2dd5d207840b31cc59083c8f4c4718959149e0358c0364052bb9a0330', 'model.safetensors-00001-of-00004.safetensors': 'db6f444b43d318c92f360a13a25561a6a65b10c0631b8ed305a426dbaa6c380e', 'model.safetensors-00004-of-00004.safetensors': 'b62b0c4cd7e44edee103ee8f4fe225f246d5e768e07bfd5f25b63a8aa1fdd0c6', 'model.safetensors.index.json': '26d3539b516be613f39563617cb9d33b3f83d401298125be392c80cefb8f7fe5', 'preprocessor_config.json': '27225450ac9c6529872ee1924fcb0962ff5634834f817040f444118116f4e516', 'tokenizer.json': '5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42', 'tokenizer_config.json': '316230d6a809701f4db5ea8f8fc862bc3a6f3229c937c174e674ff3ca0a64ac8', 'video_preprocessor_config.json': '7768af27c1fafa9cc9011c1dc20067e03f8915e03b63504550e11d5066986d13', 'vocab.json': 'ce99b4cb2983d118806ce0a8b777a35b093e2000a503ebde25853284c9dfa003', 'model.safetensors-00003-of-00004.safetensors': '7ec36ba3a4176a44c3c0876ad80c56a2f70c84bf008d82e9501df642f17dadec'}}

In [ ]:
from pathlib import Path
import base64,hashlib,json,zipfile,io,subprocess,sys,datetime
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='2889f2f5a8979253ba6a4f8ecce63f3fc18edda08226d449ca34b4d8358afcb1'
workspace=Path('/kaggle/working/relearning_v1_')/datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
workspace.mkdir(parents=True,exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert sum(i.file_size for i in z.infolist())<10*1024*1024
    for i in z.infolist():
        assert (workspace/i.filename).resolve().is_relative_to(workspace.resolve())
    z.extractall(workspace)
result=subprocess.run([sys.executable,'-X','utf8','run_checks.py'],cwd=workspace,capture_output=True,text=True)
print(result.stdout)
assert result.returncode==0,'CPU tests failed: see private TEST_DETAILS.txt; no GPU/training started'
demo=subprocess.run([sys.executable,'synthetic_demo.py'],cwd=workspace,capture_output=True,text=True)
print(demo.stdout)
assert demo.returncode==0,'Synthetic loop failed'


In [ ]:
# Additional CPU-only validators. No GPU model is loaded here.
import base64,hashlib,json,subprocess,sys
extra=base64.b64decode('eyJzbW9rZV92YWxpZGF0aW9uLnB5IjogIlwiXCJcIlBvc3QtdHJhaW5pbmcgY2hlY2tzLiBEb2VzIG5vdCBsb2FkIEdQVSBtb2RlbHMgb3IgcGlja2xlIHN0YXRlIGZpbGVzLlwiXCJcIlxuaW1wb3J0IG1hdGhcblxuZGVmIHRlbnNvcl9kZWx0YShwYXJlbnQsY2FuZGlkYXRlKTpcbiAgICBpbXBvcnQgbnVtcHkgYXMgbnBcbiAgICBpZiBzZXQocGFyZW50KSE9c2V0KGNhbmRpZGF0ZSkgb3Igbm90IHBhcmVudDpyYWlzZSBWYWx1ZUVycm9yKCdBZGFwdGVyIGtleXMgbWlzbWF0Y2gnKVxuICAgIGNoYW5nZWQ9MDtkZWx0YV9zcT0wLjtjb3VudD0wXG4gICAgZm9yIG5hbWUgaW4gcGFyZW50OlxuICAgICAgICBhPW5wLmFzYXJyYXkocGFyZW50W25hbWVdLGR0eXBlPW5wLmZsb2F0NjQpO2I9bnAuYXNhcnJheShjYW5kaWRhdGVbbmFtZV0sZHR5cGU9bnAuZmxvYXQ2NClcbiAgICAgICAgaWYgYS5zaGFwZSE9Yi5zaGFwZTpyYWlzZSBWYWx1ZUVycm9yKCdBZGFwdGVyIHRlbnNvciBzaGFwZSBtaXNtYXRjaCcpXG4gICAgICAgIGlmIG5vdCBucC5pc2Zpbml0ZShhKS5hbGwoKSBvciBub3QgbnAuaXNmaW5pdGUoYikuYWxsKCk6cmFpc2UgVmFsdWVFcnJvcignTm9uZmluaXRlIGFkYXB0ZXIgd2VpZ2h0cycpXG4gICAgICAgIGQ9Yi1hO2RlbHRhX3NxKz1mbG9hdChucC5zdW0oZCpkKSk7Y2hhbmdlZCs9aW50KG5wLmNvdW50X25vbnplcm8oZCkpO2NvdW50Kz1hLnNpemVcbiAgICBpZiBub3QgY2hhbmdlZCBvciBub3QgbWF0aC5pc2Zpbml0ZShkZWx0YV9zcSk6cmFpc2UgVmFsdWVFcnJvcignTm8gZmluaXRlIGxlYXJuZWQgd2VpZ2h0IHVwZGF0ZScpXG4gICAgcmV0dXJuIHsnYWxsX3dlaWdodHNfZmluaXRlJzpUcnVlLCdjaGFuZ2VkX2VsZW1lbnRzJzpjaGFuZ2VkLCd0b3RhbF9lbGVtZW50cyc6Y291bnQsJ2RlbHRhX2wyJzptYXRoLnNxcnQoZGVsdGFfc3EpfVxuXG5kZWYgc2FmZV9hZGFwdGVyX3RlbnNvcnMocGF0aCk6XG4gICAgZnJvbSBzYWZldGVuc29ycy50b3JjaCBpbXBvcnQgbG9hZF9maWxlXG4gICAgIyBTYWZldGVuc29ycyBvbmx5LiBOZXZlciB0b3JjaC5sb2FkIG9wdGltaXplci9STkcgZmlsZXMgZm9yIHRoaXMgY2hlY2suXG4gICAgcmV0dXJuIHtrOnYuZmxvYXQoKS5jcHUoKS5udW1weSgpIGZvciBrLHYgaW4gbG9hZF9maWxlKHN0cihwYXRoKSxkZXZpY2U9J2NwdScpLml0ZW1zKCl9XG5cbmRlZiB2YWxpZGF0ZV9hZnRlcl90cmFpbmluZyhwYXJlbnQsY2hlY2twb2ludCxjaGVja3BvaW50X3ZlcmlmaWVyLHJlbG9hZF9mb3J3YXJkKTpcbiAgICBcIlwiXCJyZWxvYWRfZm9yd2FyZCBtdXN0IGFjdHVhbGx5IHJlbG9hZCB0aGUgc2F2ZWQgYWRhcHRlciBhbmQgcmV0dXJuIGV2aWRlbmNlLlxuXG4gICAgVGhpcyBmdW5jdGlvbiBkb2VzIG5vdCBmYWJyaWNhdGUgcmVsb2FkIHJlc3VsdHMsIGFsdGVyIGEgcnVuIG1hbmlmZXN0LCBvclxuICAgIHVwZGF0ZSBCRVNUX01PREVML0xBVEVTVF9TWU5DRUQuIFdyaXRlIGl0cyByZWNlaXB0IG91dHNpZGUgdGhlIHNlYWxlZCBydW4uXG4gICAgXCJcIlwiXG4gICAgbWFuaWZlc3Rfc2hhPWNoZWNrcG9pbnRfdmVyaWZpZXIoY2hlY2twb2ludC5wYXJlbnQpXG4gICAgZGVsdGE9dGVuc29yX2RlbHRhKHNhZmVfYWRhcHRlcl90ZW5zb3JzKHBhcmVudC8nYWRhcHRlcl9tb2RlbC5zYWZldGVuc29ycycpLHNhZmVfYWRhcHRlcl90ZW5zb3JzKGNoZWNrcG9pbnQvJ2FkYXB0ZXJfbW9kZWwuc2FmZXRlbnNvcnMnKSlcbiAgICBmb3J3YXJkPXJlbG9hZF9mb3J3YXJkKGNoZWNrcG9pbnQpXG4gICAgaWYgZm9yd2FyZC5nZXQoJ3JlbG9hZGVkX2Zyb21fc2F2ZWRfYWRhcHRlcicpIGlzIG5vdCBUcnVlIG9yIG5vdCBtYXRoLmlzZmluaXRlKGZvcndhcmQuZ2V0KCdsb3NzJyxmbG9hdCgnbmFuJykpKSBvciBmb3J3YXJkLmdldCgnbWFza19zaGlmdF92ZXJpZmllZCcpIGlzIG5vdCBUcnVlOlxuICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdDaGVja3BvaW50IHJlbG9hZC9mb3J3YXJkL21hc2sgZXZpZGVuY2UgZmFpbGVkJylcbiAgICByZXR1cm4geydjaGVja3BvaW50X21hbmlmZXN0X3NoYTI1Nic6bWFuaWZlc3Rfc2hhLCdhZGFwdGVyX2RlbHRhJzpkZWx0YSwncmVsb2FkJzpmb3J3YXJkLCdzdGF0dXMnOidwYXNzZWQnLCdjaGVja3BvaW50X2Z1bGxfcmVzdW1lX3ZlcmlmaWVkJzpGYWxzZX1cblxuZGVmIHJlbG9hZF9mb3J3YXJkX2NoZWNrcG9pbnQoY2hlY2twb2ludCxiYXNlLHJvd3MpOlxuICAgIFwiXCJcIkFjdHVhbCBvZmZsaW5lIFBFRlQgcmVsb2FkICsgbmF0aXZlL21hc2tlZCBsb3NzIHBhcml0eSBvbiBhIHRpbnkgZml4dHVyZS5cblxuICAgIENhbGwgb25seSBhZnRlciBwZXJtaXNzaW9ucywgQ1VEQS9kZXBlbmRlbmNpZXMgYW5kIEdQVSBidWRnZXQgZ2F0ZXMgcGFzcy5cbiAgICBDYWxsZXIgbXVzdCByZWxlYXNlIHRoZSB0cmFpbmluZyBtb2RlbCBhbmQgc3RvcCB0aGUgR1BVIHNlc3Npb24gYWZ0ZXJ3YXJkcy5cbiAgICBcIlwiXCJcbiAgICBpbXBvcnQgZ2MsdGVtcGZpbGVcbiAgICBmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGhcbiAgICBpbXBvcnQgdG9yY2hcbiAgICBpbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGXG4gICAgZnJvbSBwZWZ0IGltcG9ydCBQZWZ0TW9kZWxcbiAgICBmcm9tIG1vZGVsaW5nIGltcG9ydCBsb2FkX2Jhc2VcbiAgICBmcm9tIHRyYWluIGltcG9ydCBDb21wbGV0aW9uRGF0YXNldCxjb2xsYXRlXG4gICAgZ2MuY29sbGVjdCgpO3RvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKVxuICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KHByZWZpeD0nemFpY19yZWxvYWRfJykgYXMgc2NyYXRjaDpcbiAgICAgICAgcHJvamVjdD1QYXRoKHNjcmF0Y2gpOyhwcm9qZWN0Lydhc3NldHMnKS5ta2RpcigpOyhwcm9qZWN0Lydhc3NldHMvYmFzZScpLnN5bWxpbmtfdG8oUGF0aChiYXNlKS5yZXNvbHZlKCksdGFyZ2V0X2lzX2RpcmVjdG9yeT1UcnVlKVxuICAgICAgICBtb2RlbCx0b2s9bG9hZF9iYXNlKHByb2plY3QsRmFsc2UpXG4gICAgICAgIG1vZGVsPVBlZnRNb2RlbC5mcm9tX3ByZXRyYWluZWQobW9kZWwsc3RyKGNoZWNrcG9pbnQpLGlzX3RyYWluYWJsZT1GYWxzZSxsb2NhbF9maWxlc19vbmx5PVRydWUpXG4gICAgICAgIG1vZGVsLmV2YWwoKTttb2RlbC5jb25maWcudXNlX2NhY2hlPUZhbHNlXG4gICAgICAgIGRzPUNvbXBsZXRpb25EYXRhc2V0KHJvd3MsdG9rLDIwNDgpXG4gICAgICAgIGlmIGxlbihkcykhPWxlbihyb3dzKSBvciBkcy5za2lwcGVkIG9yIG5vdCBsZW4oZHMpOnJhaXNlIFZhbHVlRXJyb3IoJ1JlbG9hZCBmaXh0dXJlIGludmFsaWQnKVxuICAgICAgICBiYXRjaD17azp2LnRvKCdjdWRhOjAnKSBmb3Igayx2IGluIGNvbGxhdGUoW2RzWzBdXSkuaXRlbXMoKX07bGFiZWxzPWJhdGNoLnBvcCgnbGFiZWxzJylcbiAgICAgICAgbj1pbnQobGFiZWxzLm5lKC0xMDApLnN1bSgpLml0ZW0oKSlcbiAgICAgICAgaWYgbjw9MCBvciBub3QgbGFiZWxzWzosOi1uXS5lcSgtMTAwKS5hbGwoKSBvciBub3QgbGFiZWxzWzosLW46XS5nZSgwKS5hbGwoKTpyYWlzZSBWYWx1ZUVycm9yKCdDb21wbGV0aW9uIG1hc2sgaW52YWxpZCcpXG4gICAgICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKTpcbiAgICAgICAgICAgIHJlZmVyZW5jZT1tb2RlbCgqKmJhdGNoLGxhYmVscz1sYWJlbHMsdXNlX2NhY2hlPUZhbHNlKVxuICAgICAgICAgICAgbmF0aXZlX2xvc3M9ZmxvYXQocmVmZXJlbmNlLmxvc3MuaXRlbSgpKTtkZWwgcmVmZXJlbmNlXG4gICAgICAgICAgICBjb21wYWN0PW1vZGVsKCoqYmF0Y2gsbG9naXRzX3RvX2tlZXA9bisxLHVzZV9jYWNoZT1GYWxzZSlcbiAgICAgICAgICAgIGxvZ2l0cz1jb21wYWN0LmxvZ2l0c1s6LDotMSw6XS5mbG9hdCgpXG4gICAgICAgICAgICBsb3NzPWZsb2F0KEYuY3Jvc3NfZW50cm9weShsb2dpdHMucmVzaGFwZSgtMSxsb2dpdHMuc2hhcGVbLTFdKSxsYWJlbHNbOiwtbjpdLnJlc2hhcGUoLTEpKS5pdGVtKCkpXG4gICAgICAgIGlmIG5vdCBtYXRoLmlzZmluaXRlKGxvc3MpIG9yIG5vdCBtYXRoLmlzZmluaXRlKG5hdGl2ZV9sb3NzKSBvciBhYnMobG9zcy1uYXRpdmVfbG9zcyk+LjAyOnJhaXNlIFZhbHVlRXJyb3IoJ05hdGl2ZS9jb21wYWN0IGxvc3MgcGFyaXR5IGZhaWxlZCcpXG4gICAgICAgIHJlc3VsdD17J3JlbG9hZGVkX2Zyb21fc2F2ZWRfYWRhcHRlcic6VHJ1ZSwnbG9zcyc6bG9zcywnbmF0aXZlX2xvc3MnOm5hdGl2ZV9sb3NzLCdsb3NzX2RlbHRhJzphYnMobG9zcy1uYXRpdmVfbG9zcyksJ21hc2tfc2hpZnRfdmVyaWZpZWQnOlRydWUsJ2lucHV0X3Rva2Vucyc6aW50KGJhdGNoWydpbnB1dF9pZHMnXS5zaGFwZVsxXSksJ2NvbXBsZXRpb25fdG9rZW5zJzpufVxuICAgICAgICBkZWwgbW9kZWwsYmF0Y2gsbGFiZWxzLGNvbXBhY3QsbG9naXRzXG4gICAgICAgIGdjLmNvbGxlY3QoKTt0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKClcbiAgICAgICAgcmV0dXJuIHJlc3VsdFxuIiwgInRlc3Rfc21va2VfdmFsaWRhdGlvbi5weSI6ICJpbXBvcnQgdW5pdHRlc3RcbmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aFxuZnJvbSB1bml0dGVzdC5tb2NrIGltcG9ydCBwYXRjaCxNb2NrXG5mcm9tIHNtb2tlX3ZhbGlkYXRpb24gaW1wb3J0IHRlbnNvcl9kZWx0YSx2YWxpZGF0ZV9hZnRlcl90cmFpbmluZ1xuY2xhc3MgU21va2VUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6XG4gICAgZGVmIHRlc3RfZmluaXRlX3VwZGF0ZShzZWxmKTpcbiAgICAgICAgcj10ZW5zb3JfZGVsdGEoeydhJzpbMS4sMi5dfSx7J2EnOlsxLiwyLjFdfSk7c2VsZi5hc3NlcnRFcXVhbChyWydjaGFuZ2VkX2VsZW1lbnRzJ10sMSk7c2VsZi5hc3NlcnRHcmVhdGVyKHJbJ2RlbHRhX2wyJ10sMClcbiAgICBkZWYgdGVzdF91bmNoYW5nZWRfcmVqZWN0ZWQoc2VsZik6XG4gICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6dGVuc29yX2RlbHRhKHsnYSc6WzEuXX0seydhJzpbMS5dfSlcbiAgICBkZWYgdGVzdF9uYW5fcmVqZWN0ZWQoc2VsZik6XG4gICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6dGVuc29yX2RlbHRhKHsnYSc6WzEuXX0seydhJzpbZmxvYXQoJ25hbicpXX0pXG4gICAgZGVmIHRlc3Rfd3Jvbmdfa2V5c19yZWplY3RlZChzZWxmKTpcbiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKTp0ZW5zb3JfZGVsdGEoeydhJzpbMS5dfSx7J2InOlsxLl19KVxuICAgIGRlZiB0ZXN0X3dyb25nX3NoYXBlX3JlamVjdGVkKHNlbGYpOlxuICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOnRlbnNvcl9kZWx0YSh7J2EnOlsxLl19LHsnYSc6WzEuLDIuXX0pXG4gICAgZGVmIHRlc3RfcmVsb2FkX3JlY2VpcHRfcmVxdWlyZXNfcmVhbF9mbGFnX2FuZF9sb3NzKHNlbGYpOlxuICAgICAgICB3aXRoIHBhdGNoKCdzbW9rZV92YWxpZGF0aW9uLnNhZmVfYWRhcHRlcl90ZW5zb3JzJyxzaWRlX2VmZmVjdD1beydhJzpbMS5dfSx7J2EnOlsyLl19XSk6XG4gICAgICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOnZhbGlkYXRlX2FmdGVyX3RyYWluaW5nKFBhdGgoJ3BhcmVudCcpLFBhdGgoJ3J1bi9jaGVja3BvaW50JyksbGFtYmRhIHA6J3NoYScsbGFtYmRhIHA6eydsb3NzJzpmbG9hdCgnaW5mJyksJ21hc2tfc2hpZnRfdmVyaWZpZWQnOlRydWUsJ3JlbG9hZGVkX2Zyb21fc2F2ZWRfYWRhcHRlcic6VHJ1ZX0pXG4gICAgZGVmIHRlc3RfdmFsaWRfcmVjZWlwdF9kb2VzX25vdF9jbGFpbV9mdWxsX3Jlc3VtZShzZWxmKTpcbiAgICAgICAgd2l0aCBwYXRjaCgnc21va2VfdmFsaWRhdGlvbi5zYWZlX2FkYXB0ZXJfdGVuc29ycycsc2lkZV9lZmZlY3Q9W3snYSc6WzEuXX0seydhJzpbMi5dfV0pOlxuICAgICAgICAgICAgcmVzdWx0PXZhbGlkYXRlX2FmdGVyX3RyYWluaW5nKFBhdGgoJ3BhcmVudCcpLFBhdGgoJ3J1bi9jaGVja3BvaW50JyksbGFtYmRhIHA6J3NoYScsbGFtYmRhIHA6eydsb3NzJzoxLiwnbWFza19zaGlmdF92ZXJpZmllZCc6VHJ1ZSwncmVsb2FkZWRfZnJvbV9zYXZlZF9hZGFwdGVyJzpUcnVlfSlcbiAgICAgICAgc2VsZi5hc3NlcnRGYWxzZShyZXN1bHRbJ2NoZWNrcG9pbnRfZnVsbF9yZXN1bWVfdmVyaWZpZWQnXSk7c2VsZi5hc3NlcnRFcXVhbChyZXN1bHRbJ3N0YXR1cyddLCdwYXNzZWQnKVxuICAgIGRlZiB0ZXN0X2JhZF9tYW5pZmVzdF9zdG9wc19iZWZvcmVfcmVsb2FkKHNlbGYpOlxuICAgICAgICByZWxvYWQ9TW9jaygpXG4gICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6dmFsaWRhdGVfYWZ0ZXJfdHJhaW5pbmcoUGF0aCgncGFyZW50JyksUGF0aCgncnVuL2NoZWNrcG9pbnQnKSxNb2NrKHNpZGVfZWZmZWN0PVZhbHVlRXJyb3IoKSkscmVsb2FkKVxuICAgICAgICByZWxvYWQuYXNzZXJ0X25vdF9jYWxsZWQoKVxuaWYgX19uYW1lX189PSdfX21haW5fXyc6dW5pdHRlc3QubWFpbigpXG4ifQ==')
assert hashlib.sha256(extra).hexdigest()=='b8b248385a9b3d6c58166d16008ef123838a9f87938ebfdd188c1a106d0a2057'
for name,source_text in json.loads(extra).items():
    target=workspace/name
    assert target.resolve().is_relative_to(workspace.resolve())
    with target.open('x',encoding='utf-8') as f:f.write(source_text)
check=subprocess.run([sys.executable,'-X','utf8','-m','unittest','test_smoke_validation'],cwd=workspace,capture_output=True,text=True)
assert check.returncode==0,'Additional CPU smoke validators failed; do not enable GPU'
print('Additional smoke validators: 8 CPU tests passed; no GPU execution')


In [ ]:
# Default remains preflight. Complete evidence and CPU input checks before GPU.
if CONFIG['action']=='gpu_smoke':
    assert CONFIG['gpu_confirmed'], 'Review limited smoke budget first'
    assert CONFIG['competition_permission_evidence']['status']=='verified', 'Competition evidence is still unverified'
    assert CONFIG['max_steps']==1, 'This notebook only permits one smoke optimizer step'
    from pathlib import Path
    bases=list(Path('/kaggle/input').rglob('BASE_UPLOAD_MANIFEST.json'))
    adapters=list(Path('/kaggle/input').rglob('final/adapter_model.safetensors'))
    assert len(bases)==len(adapters)==1
    assert CONFIG['base_hashes']=={k:v['sha256'] for k,v in json.loads(bases[0].read_text())['files'].items()}
    CONFIG['base']=str(bases[0].parent/'base')
    CONFIG['adapter']=str(adapters[0].parent)
    CONFIG['output_root']='/kaggle/working/relearning_model_runs'
    request=workspace/'GPU_SMOKE_REQUEST.json'
    request.write_text(json.dumps(CONFIG,indent=2))
    child=workspace/'execute_gpu_smoke.py'
    with child.open('x',encoding='utf-8') as f:
        f.write("from pathlib import Path\nimport sys,json\nfrom train_warm import train\nfrom smoke_targets import targets\nfrom relearning import verify_export,atomic\nfrom smoke_validation import validate_after_training,reload_forward_checkpoint\nconfig=json.loads(Path(sys.argv[1]).read_text())\nrows=targets()\nrun=train(config,rows)\nreceipt=validate_after_training(Path(config['adapter']),run/'checkpoint',verify_export,lambda ck:reload_forward_checkpoint(ck,Path(config['base']),rows))\nreceipt.update(run_id=run.name,init_mode='warm_start',steps=1,gpu_session_stop_required=True)\natomic(Path('GPU_SMOKE_RECEIPT.json'),receipt)\n")
    # Bound the child process, including model load, training and reload.
    # Stopping the Kaggle GPU session itself is still an explicit GUI action.
    try:
        with (workspace/'GPU_SMOKE_PRIVATE.log').open('w',encoding='utf-8') as log:
            job=subprocess.run([sys.executable,str(child),str(request)],cwd=workspace,stdout=log,stderr=subprocess.STDOUT,timeout=900)
        assert job.returncode==0,'GPU smoke failed: retain private log and stop GPU session now'
        print((workspace/'GPU_SMOKE_RECEIPT.json').read_text())
    finally:
        print('STOP SESSION NOW, set Accelerator=None and verify off. Download saved output afterwards.')
elif CONFIG['action']!='preflight':
    raise ValueError('Long training is disabled in this smoke notebook')
